# Lab #7: Keras MLP for Multiclass Classification

**Dataset used:** Wine dataset (built into scikit-learn) — 178 wine samples, 13 chemical-analysis features, 3 classes (3 different wine cultivars). It's a real, multiclass, tabular dataset that needs zero downloading, making this notebook fully self-contained and application-oriented.

## Objective

Implement a Multi-Layer Perceptron (MLP) using Keras/TensorFlow for a multiclass classification problem, and analyze the effect of different **activation functions** and **optimizers** on classification performance.

## Problem Statement

Predict which of 3 wine cultivars a sample belongs to, based on 13 chemical measurements (alcohol content, malic acid, ash, flavanoids, color intensity, etc.). This is a 3-class classification problem — a good multiclass counterpart to the binary/Iris problems from earlier labs.

### Importing the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam, RMSprop

tf.__version__

## 1. Dataset Preparation

### Load and explore the dataset

In [ ]:
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
class_names = wine.target_names

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print('Number of samples :', X.shape[0])
print('Number of features:', X.shape[1])
print('Number of classes :', len(class_names))
print('Class names       :', list(class_names))
df.head()

### Class distribution

In [ ]:
class_counts = df['target'].value_counts().sort_index()
print(class_counts)

plt.figure(figsize=(5, 4))
sns.barplot(x=class_names, y=class_counts.values)
plt.title('Class Distribution - Wine Dataset')
plt.xlabel('Wine Cultivar')
plt.ylabel('Number of Samples')
plt.show()

### Handle missing values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum().sum(), 'total missing values')
# The sklearn Wine dataset is already clean, but we check anyway as good practice

### Encode the target variable (one-hot encoding for Softmax)

In [ ]:
y_categorical = to_categorical(y, num_classes=len(class_names))
print('Example label before encoding:', y[0])
print('Example label after encoding :', y_categorical[0])

### Train/test split and feature scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling: the 13 features are on very different scales (e.g. proline is in the
# hundreds, hue is <2), so StandardScaler is essential here for the MLP to train well.
# Fit ONLY on training data, then apply the same transform to test data to avoid data leakage.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Training samples:', X_train.shape[0])
print('Test samples    :', X_test.shape[0])

**Justification of preprocessing:** Standard scaling is used (not min-max) because the features are roughly normally distributed and StandardScaler is more robust to outliers than min-max scaling. One-hot encoding is required because Softmax output expects a probability vector per class, not a single integer label. A stratified 80/20 split keeps class proportions consistent between train and test sets, which matters since we only have 178 samples total. A validation split (10% of training data) is used during `model.fit()` as the validation strategy, so the test set stays fully unseen until final evaluation.

## 2. MLP Model Development

**Architecture:** Input(13) → Dense(16, hidden activation) → Dense(8, hidden activation) → Output Dense(3, Softmax)

- **Input features:** 13
- **Number of classes:** 3
- **Hidden layer 1:** 16 neurons
- **Hidden layer 2:** 8 neurons
- **Output activation:** Softmax (required for multiclass probability output)
- **Loss function:** Categorical Crossentropy
- **Batch size:** 8
- **Epochs:** 50
- **Validation split:** 10% of training data

The hidden-layer activation and optimizer are kept as parameters below so we can swap them cleanly for Experiments A and B without duplicating code.

In [ ]:
def build_model(hidden_activation='relu', optimizer='adam', input_dim=13, num_classes=3):
    model = Sequential([
        Dense(16, activation=hidden_activation, input_shape=(input_dim,)),
        Dense(8, activation=hidden_activation),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Quick check of the architecture
build_model().summary()

## Helper function: train + evaluate one configuration
Used by both Experiment A (activation functions) and Experiment B (optimizers) so every model is trained and evaluated in exactly the same way — same split, same epochs, same batch size — for a fair comparison.

In [ ]:
results = {}  # stores metrics for every experiment, used later for the comparison table
histories = {}  # stores training history for plotting

def run_experiment(name, hidden_activation='relu', optimizer='adam', epochs=50, batch_size=8):
    print(f'--- Training: {name} (activation={hidden_activation}, optimizer={optimizer}) ---')
    model = build_model(hidden_activation=hidden_activation, optimizer=optimizer)

    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.1,
                        verbose=0)

    # Evaluate on the held-out test set
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test, axis=1)

    test_acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')

    results[name] = {
        'activation': hidden_activation,
        'optimizer': optimizer,
        'test_accuracy': test_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'final_train_acc': history.history['accuracy'][-1],
        'final_val_acc': history.history['val_accuracy'][-1],
        'final_train_loss': history.history['loss'][-1],
        'final_val_loss': history.history['val_loss'][-1],
    }
    histories[name] = history

    print(f'Test Accuracy: {test_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}')
    return model, history

## 3. Experiment A - Hidden-Layer Activation Functions

We keep the optimizer fixed at **Adam** and compare three hidden-layer activation functions: **ReLU**, **Sigmoid**, and **Tanh**. Everything else (architecture, epochs, batch size, split) stays identical, so any difference in performance is attributable to the activation function alone.

In [ ]:
_ = run_experiment('ReLU + Adam', hidden_activation='relu', optimizer='adam')
_ = run_experiment('Sigmoid + Adam', hidden_activation='sigmoid', optimizer='adam')
_ = run_experiment('Tanh + Adam', hidden_activation='tanh', optimizer='adam')

### Plot: Activation Function Comparison

In [ ]:
activation_models = ['ReLU + Adam', 'Sigmoid + Adam', 'Tanh + Adam']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for name in activation_models:
    h = histories[name].history
    axes[0,0].plot(h['accuracy'], label=name)
    axes[0,1].plot(h['val_accuracy'], label=name)
    axes[1,0].plot(h['loss'], label=name)
    axes[1,1].plot(h['val_loss'], label=name)

axes[0,0].set_title('Training Accuracy vs Epoch'); axes[0,0].set_xlabel('Epoch'); axes[0,0].legend()
axes[0,1].set_title('Validation Accuracy vs Epoch'); axes[0,1].set_xlabel('Epoch'); axes[0,1].legend()
axes[1,0].set_title('Training Loss vs Epoch'); axes[1,0].set_xlabel('Epoch'); axes[1,0].legend()
axes[1,1].set_title('Validation Loss vs Epoch'); axes[1,1].set_xlabel('Epoch'); axes[1,1].legend()

plt.tight_layout()
plt.show()

**Why activation functions may behave differently here:** ReLU avoids the vanishing-gradient problem that Sigmoid and Tanh can suffer from in deeper networks, so it typically converges faster. Sigmoid squashes outputs into (0,1) and can saturate (gradients close to zero) when inputs are large in magnitude, slowing learning. Tanh is zero-centered (range -1 to 1) so it often trains a bit better than Sigmoid, but can still saturate. With only 2 hidden layers here the gap may be modest, but it should still be visible in the plots above.

## 4. Experiment B - Optimizer Comparison

We now fix the hidden-layer activation at **ReLU** (the strongest performer from Experiment A in most cases) and compare three optimizers: **SGD**, **Adam**, and **RMSprop**. Architecture, epochs, batch size, and split remain identical.

In [ ]:
_ = run_experiment('ReLU + SGD', hidden_activation='relu', optimizer=SGD(learning_rate=0.01))
_ = run_experiment('ReLU + RMSprop', hidden_activation='relu', optimizer=RMSprop(learning_rate=0.001))
# 'ReLU + Adam' was already trained in Experiment A and is reused here for the comparison

### Plot: Optimizer Comparison

In [ ]:
optimizer_models = ['ReLU + SGD', 'ReLU + Adam', 'ReLU + RMSprop']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for name in optimizer_models:
    h = histories[name].history
    axes[0,0].plot(h['accuracy'], label=name)
    axes[0,1].plot(h['val_accuracy'], label=name)
    axes[1,0].plot(h['loss'], label=name)
    axes[1,1].plot(h['val_loss'], label=name)

axes[0,0].set_title('Training Accuracy vs Epoch'); axes[0,0].set_xlabel('Epoch'); axes[0,0].legend()
axes[0,1].set_title('Validation Accuracy vs Epoch'); axes[0,1].set_xlabel('Epoch'); axes[0,1].legend()
axes[1,0].set_title('Training Loss vs Epoch'); axes[1,0].set_xlabel('Epoch'); axes[1,0].legend()
axes[1,1].set_title('Validation Loss vs Epoch'); axes[1,1].set_xlabel('Epoch'); axes[1,1].legend()

plt.tight_layout()
plt.show()

**Why these optimizers behave differently:** SGD uses a single fixed learning rate for all weights and no momentum by default, so it tends to converge slower and can be noisier. Adam adapts the learning rate per-parameter using estimates of first and second moments of the gradients, generally giving fast, stable convergence with little tuning — which is why it's a strong default choice. RMSprop also adapts the learning rate per-parameter (using a moving average of squared gradients) and often performs similarly to Adam, converging faster than plain SGD.

## 5. Model Training

All 5 models above (`ReLU+Adam`, `Sigmoid+Adam`, `Tanh+Adam`, `ReLU+SGD`, `ReLU+RMSprop`) were trained using the **same** train/test split, the **same** architecture (16 → 8 → 3 neurons), the **same** number of epochs (50), batch size (8), and validation split (10%) — the Training/Validation Accuracy and Loss vs Epoch plots for each are shown above in Experiments A and B.

## 6. Model Evaluation

We now take a closer look at the best-performing model (based on test accuracy from the results collected so far) with a full evaluation: classification report and confusion matrix.

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['test_accuracy'])
print('Best performing model so far:', best_model_name)
print(results[best_model_name])

In [ ]:
# Retrain the best configuration once more to get a live model object for evaluation
best_activation = results[best_model_name]['activation']
best_optimizer_name = results[best_model_name]['optimizer']

best_model, best_history = run_experiment('Best Model (' + best_model_name + ')',
                                          hidden_activation=best_activation,
                                          optimizer=best_optimizer_name if isinstance(best_optimizer_name, str) else 'adam')

y_pred_probs = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

**Interpreting the confusion matrix:** the diagonal cells show correct predictions per class; any off-diagonal (non-zero) cell shows a misclassification — e.g., a value at row `class_1`, column `class_2` means some `class_1` samples were predicted as `class_2`. Look at which diagonal value is highest (most accurately predicted class) and which off-diagonal cells are non-zero (most confused pair) once you run this cell — fill in your specific findings here.

## 7. Comparison Table

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df[['activation', 'optimizer', 'test_accuracy', 'precision', 'recall', 'f1_score']]
comparison_df.columns = ['Activation', 'Optimizer', 'Test Accuracy', 'Precision', 'Recall', 'F1-Score']
comparison_df = comparison_df.round(4)
comparison_df

## 8. Analysis and Interpretation

Once you've run the notebook, use the comparison table and plots above to answer:

1. **Which activation function performed best? Why?** — Compare the `test_accuracy`/`f1_score` rows for ReLU, Sigmoid, and Tanh (all with Adam) in the table.
2. **Which optimizer performed best? Why?** — Compare the ReLU rows across SGD, Adam, and RMSprop.
3. **Which activation+optimizer combination was best overall?** — The row with the highest Test Accuracy / F1-Score.
4. **Did activation function significantly affect convergence?** — Check how quickly training/validation accuracy rose in the Experiment A plots.
5. **Did optimizer affect convergence speed?** — Check the Experiment B plots; SGD usually converges more slowly than Adam/RMSprop.
6. **Which model achieved the best validation performance?** — Compare `final_val_acc` across all 5 entries in `results`.
7. **Which model achieved the best test performance?** — The `best_model_name` computed above.
8. **Is there a big gap between training and test performance?** — Compare `final_train_acc` vs `test_accuracy` per model; a large gap suggests overfitting.
9. **Overfitting or underfitting?** — If training accuracy is high but validation accuracy plateaus much lower, that's overfitting; if both stay low, that's underfitting.
10–11. **Most misclassified class(es) and why?** — From the confusion matrix; often due to overlapping chemical profiles between two wine cultivars.
12. **What could improve performance?** — More epochs with early stopping, more hidden units, dropout regularization, or more training data.
13. **Final model of choice and why?** — See Section 9 below.

*(Fill in your specific numeric observations here once you've run the notebook end to end.)*

## 9. Final Model Selection

In [ ]:
final_choice = comparison_df['F1-Score'].astype(float).idxmax()
print('Suggested final model based on F1-Score:', final_choice)
comparison_df.loc[[final_choice]]

**Justification:** the final model isn't picked on test accuracy alone — F1-score is used here since it balances precision and recall, which matters if any class has slightly fewer samples. The chosen model should also show validation loss/accuracy tracking training loss/accuracy closely in its plot (stable convergence, not overfitting), and a clean confusion matrix with few off-diagonal errors. Model complexity is also modest here (16→8 neurons) for all 5 candidates, so complexity was not a deciding factor in this comparison — the difference came down to how well each activation/optimizer combination converged.

## Conclusion

This lab implemented a Keras/TensorFlow MLP for multiclass classification on the Wine dataset (3 classes, 13 features), comparing three hidden-layer activation functions (ReLU, Sigmoid, Tanh) and three optimizers (SGD, Adam, RMSprop) under identical train/test splits and architectures. Using accuracy, precision, recall, F1-score, and confusion matrices, the best-performing configuration was identified and selected as the final model, demonstrating how activation function and optimizer choices materially affect convergence speed and generalization even when the underlying architecture stays the same.